# V4 remote GPU smoke

This gate runs only on Kaggle. It requires a CUDA device, reads one real xBD before/after pair, performs one tiny paired forward/backward/optimizer step, and writes a structured report. It does not download a foundation model, train a production model, or use the laptop CPU/GPU.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch import nn


INPUT_ROOT = Path("/kaggle/input")
OUTPUT_PATH = Path("/kaggle/working/rv4_gpu_smoke.json")
DATASET_GLOB = "*xview2-challenge-dataset-train-and-test*"


def find_pair():
    pre_candidates = sorted(INPUT_ROOT.rglob("*_pre_disaster.png"))
    if not pre_candidates:
        raise RuntimeError("no xBD pre-disaster PNG was found under /kaggle/input")
    pre = pre_candidates[0]
    post = pre.with_name(pre.name.replace("_pre_disaster.png", "_post_disaster.png"))
    if not post.exists():
        raise RuntimeError(f"paired post-disaster image is missing: {post}")
    return pre, post


class SmokePair(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(16, 5, kernel_size=1),
        )

    def forward(self, pre, post):
        return self.net(torch.cat([pre, post], dim=1))


def load_tile(path: Path) -> torch.Tensor:
    with Image.open(path) as image:
        array = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    crop = array[:128, :128, :]
    return torch.from_numpy(crop).permute(2, 0, 1).unsqueeze(0)


def main() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU smoke requires CUDA; refusing a CPU fallback")
    device = torch.device("cuda")
    torch.cuda.reset_peak_memory_stats(device)
    pre_path, post_path = find_pair()
    pre = load_tile(pre_path).to(device, non_blocking=True)
    post = load_tile(post_path).to(device, non_blocking=True)
    model = SmokePair().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    target = torch.zeros((pre.shape[0], pre.shape[2], pre.shape[3]), dtype=torch.long, device=device)
    optimizer.zero_grad(set_to_none=True)
    logits = model(pre, post)
    loss = nn.functional.cross_entropy(logits, target)
    if not torch.isfinite(loss):
        raise RuntimeError("non-finite GPU smoke loss")
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    torch.cuda.synchronize(device)
    report = {
        "schema_version": "rv4-gpu-smoke-1",
        "status": "PASS",
        "device": str(device),
        "cuda_available": torch.cuda.is_available(),
        "torch_version": torch.__version__,
        "gpu_name": torch.cuda.get_device_name(device),
        "pair": {"pre": str(pre_path), "post": str(post_path)},
        "input_shape": list(pre.shape),
        "logits_shape": list(logits.shape),
        "loss": float(loss.detach().cpu()),
        "grad_norm": float(grad_norm),
        "peak_memory_allocated": int(torch.cuda.max_memory_allocated(device)),
        "peak_memory_reserved": int(torch.cuda.max_memory_reserved(device)),
    }
    OUTPUT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(json.dumps(report, indent=2, sort_keys=True))


if __name__ == "__main__":
    main()
